# `5.compile` — Compilation passes

This notebook is the MLIR-stack counterpart of
[examples/5.compile.ipynb](../../examples/5.compile.ipynb).

The legacy notebook walks a program through every layer of the qstack
compiler stack: toy → cliffords → h2, plus QEC passes (rep3 + steane).
This MLIR-stack version covers the two passes ported so far:

1. **`compile_toy_to_cliffords`** — 1:1 rewrite of toy gates to the
   Clifford ISA (`flip → x`, `mix → h`, `entangle → cx`).
2. **`compile_rep3`** — the trivial 3-bit repetition code QEC pass
   (showcased fully in [rep3_demo.ipynb](rep3_demo.ipynb)).

The H2-native lowering (`cliffords2h2`) and Steane CSS encoding are
deferred — they require a new dialect and a substantially larger pass
respectively. See `mlir/DESIGN.md` for the long-term plan.

In [ ]:
%load_ext qstack_mlir.jupyter

## 1. Source program in the toy ISA

A two-qubit Bell preparation written in toy-ISA gates: `mix` plays the
role of Hadamard, `entangle` the role of CNOT.

In [ ]:
%%qasm toy_bell
QSTACKQASM 0.1;
include "qstack/toy.inc";

qreg q[2];
creg c[2];
mix q[0];
entangle q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];

In [ ]:
print(toy_bell)

## 2. Compile toy → Cliffords

`compile_toy_to_cliffords` walks the module in place, swapping each
toy gate for its Clifford equivalent. SSA threading is preserved —
every replacement op has the same operand / result shape as the one it
replaced.

In [ ]:
from qstack_mlir.passes.toy2cliffords import compile_toy_to_cliffords

compile_toy_to_cliffords(toy_bell)
print(toy_bell)

The `toy.mix` and `toy.entangle` ops have become `cliffords.h` and
`cliffords.cx`. The classical surface — `measure`, `qstack.return`,
`func.func @main` — is untouched: rewriting the quantum layer does not
disturb the host-language interface.

## 3. Run the compiled module

The emulator only ever needed to know how to dispatch ops by type, so
the compiled module runs unchanged through the same `Machine`.

In [ ]:
from qstack_mlir.runtime import Machine

machine = Machine(toy_bell, num_qubits=4)
machine.shots("main", 2000).plot_histogram()

The compiled program produces the same Bell distribution as the
toy-ISA original — semantics preserved across the rewrite.

## 4. Composing passes: Cliffords → rep3

The 3-bit repetition QEC pass (showcased in
[rep3_demo.ipynb](rep3_demo.ipynb)) takes a Cliffords module and
encodes every logical qubit as three physical qubits with a
`qstack.decode @majority_vote` callout per measurement. Because both
passes operate on the same dialect, they compose: any toy-ISA program
that is Clifford-decomposable can be lifted into the encoded space by
chaining the two passes.

See [rep3_demo.ipynb](rep3_demo.ipynb) for the full demonstration,
including self-composition into a 9-qubit concatenated code.